# MAX-3 · Maximum Model — Single-Fold Test (NHANES)

Trains the maximum configuration on the NHANES fold: CORN ordinal head, full
backbone fine-tuning, 384×384 resolution, soft ordinal targets, source-reliability
weighting, sampling, curriculum, and domain-adversarial training. Results are
written to scope3_max; the original pipeline is untouched. This single fold
confirms whether the maximum stack improves on the baseline before running all
four folds.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np, torch
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU (A100).')
print('GPU:', torch.cuda.get_device_name(0))
print('Resolution:', TM.MAX_IMG_SIZE, '| Freeze:', TM.MAX_FREEZE_STAGES, '| MRKR trust:', TM.MRKR_TRUST)
manifest = TM.prepare_local_data()
print(f'Manifest: {len(manifest):,} rows')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Resolution: 384 | Freeze: 0 | MRKR trust: 0.4
Copied array in 31s
Loaded array (61558, 224, 224) in 1s
Manifest: 61,558 rows


## Train maximum model on Mendeley fold

In [2]:
mech = {'sampler':True,'noise_loss':True,'curriculum':True,'domain_adv':True,
        'hierarchical':True,'use_quality':True,'soft_alpha':TM.MAX_SOFT_ALPHA,
        'freeze_stages':TM.MAX_FREEZE_STAGES,'grl_lambda_max':0.5}

test_ds = 'nhanes3'
tr, va, te = TM.make_splits(manifest, test_ds, seed=0)
print(f'train={len(tr):,}  val={len(va):,}  test={len(te):,}')

res = TM.run_training('max_nhanes3_seed0', tr, va, te, mech, seed=0,
                      log_fn=print)

train=48,257  val=8,516  test=4,785
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 236MB/s]


  [max_nhanes3_seed0] ep1/18 loss=1.096 tr=0.342 val=0.478 w1=0.710 qwk=0.561 gap=-0.135 (621s)
  [max_nhanes3_seed0] ep2/18 loss=0.839 tr=0.533 val=0.477 w1=0.746 qwk=0.600 gap=+0.056 (598s)
  [max_nhanes3_seed0] ep3/18 loss=0.751 tr=0.598 val=0.521 w1=0.765 qwk=0.631 gap=+0.077 (595s)
  [max_nhanes3_seed0] ep4/18 loss=0.703 tr=0.637 val=0.513 w1=0.766 qwk=0.616 gap=+0.124 (594s)
  [max_nhanes3_seed0] ep5/18 loss=0.667 tr=0.659 val=0.538 w1=0.771 qwk=0.629 gap=+0.122 (594s)
  [max_nhanes3_seed0] ep6/18 loss=0.721 tr=0.667 val=0.581 w1=0.788 qwk=0.648 gap=+0.085 (595s)
  [max_nhanes3_seed0] ep7/18 loss=2.708 tr=0.568 val=0.351 w1=0.456 qwk=0.000 gap=+0.217 (594s)
  [max_nhanes3_seed0] ep8/18 loss=3.635 tr=0.351 val=0.351 w1=0.456 qwk=0.002 gap=-0.000 (594s)
  [max_nhanes3_seed0] ep9/18 loss=3.183 tr=0.341 val=0.351 w1=0.456 qwk=0.000 gap=-0.010 (594s)
  [max_nhanes3_seed0] ep10/18 loss=2.767 tr=0.349 val=0.351 w1=0.456 qwk=0.000 gap=-0.002 (594s)
  [max_nhanes3_seed0] ep11/18 loss=2.57

## Compare to baseline (original H, Mendeley fold)

In [3]:
import json, glob
def load(p):
    try: return json.load(open(str(p)))
    except Exception: return None

h = load(config.RESULTS_DIR/'fold3_test_nhanes3_seed0.json')
h_w1 = np.nan
zf = sorted(glob.glob(str(config.RESULTS_DIR/'fold3_test_nhanes3_seed*_preds.npz')))
if zf:
    z=np.load(zf[0]); h_w1=float((np.abs(z['y_true']-z['y_pred'])<=1).mean())

print('NHANES fold — baseline H vs MAX (seed 0):')
print(f"{'metric':12s}{'H (224)':>12s}{'MAX (384)':>12s}")
if h:
    print(f"{'exact acc':12s}{h['external_acc5']:>12.3f}{res['external_acc5']:>12.3f}")
    print(f"{'within-1':12s}{h_w1:>12.3f}{res['external_within1']:>12.3f}")
    print(f"{'QWK':12s}{h['external_qwk']:>12.3f}{res['external_qwk']:>12.3f}")
    print(f"{'gap (pp)':12s}{h['gap']*100:>12.1f}{res['gap']*100:>12.1f}")
    print()
    print(f"Delta exact: {res['external_acc5']-h['external_acc5']:+.3f}")
    print(f"Delta within-1: {res['external_within1']-h_w1:+.3f}")
else:
    print(f"MAX: exact={res['external_acc5']:.3f} within1={res['external_within1']:.3f} qwk={res['external_qwk']:.3f} gap={res['gap']*100:.1f}pp")

NHANES fold — baseline H vs MAX (seed 0):
metric           H (224)   MAX (384)
exact acc          0.608       0.609
within-1           0.808       0.808
QWK                0.607       0.628
gap (pp)            -8.1        -2.8

Delta exact: +0.001
Delta within-1: -0.000
